# Data Collection

Download NYC crash data from the NYC Open Data portal and save raw CSV.

## Section 1: Download Raw Data from NYC Open Data API

In [ ]:
import pandas as pd
import requests
from pathlib import Path

# --- Config ---
RAW_DATA_DIR = Path("../data/raw")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = RAW_DATA_DIR / "nyc_crashes_raw.csv"

# NYC Open Data — Motor Vehicle Collisions (Socrata API)
# Dataset ID: h9gi-nx95
# We pull in chunks because the dataset is ~2M rows
# Socrata limits 1000 rows per request by default; we use $limit and $offset to page through

APP_TOKEN = None  # optional — add yours if you get rate limited

BASE_URL = "https://data.cityofnewyork.us/resource/h9gi-nx95.csv"

CHUNK_SIZE = 50_000   # rows per request
MAX_ROWS   = 500_000  # cap for now — enough for a solid project, saves time

print(f"Downloading up to {MAX_ROWS:,} crash records from NYC Open Data...")
print("This will take a few minutes.\n")

chunks = []
offset = 0

while offset < MAX_ROWS:
    params = {
        "$limit":  CHUNK_SIZE,
        "$offset": offset,
        "$order":  "crash_date DESC",  # most recent first
    }
    if APP_TOKEN:
        params["$$app_token"] = APP_TOKEN

    response = requests.get(BASE_URL, params=params, timeout=60)
    
    if response.status_code != 200:
        print(f"Error at offset {offset}: {response.status_code}")
        break

    chunk = pd.read_csv(pd.io.common.StringIO(response.text))
    
    if len(chunk) == 0:
        print("No more data returned — download complete.")
        break

    chunks.append(chunk)
    offset += CHUNK_SIZE
    print(f"  Downloaded {min(offset, MAX_ROWS):,} rows...")

# Combine all chunks
df = pd.concat(chunks, ignore_index=True)
print(f"\nTotal rows downloaded: {len(df):,}")
print(f"Columns: {list(df.columns)}")

# Save raw file — we never modify this file, only read from it
df.to_csv(OUTPUT_FILE, index=False)
print(f"\n✓ Saved to {OUTPUT_FILE}")

## Section 2: Load and Explore Raw Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Load the raw file
df = pd.read_csv("../data/raw/nyc_crashes_raw.csv", low_memory=False)

print("=== BASIC SHAPE ===")
print(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}")

print("\n=== DATA TYPES ===")
print(df.dtypes)

print("\n=== MISSING VALUES (columns with any nulls) ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print(missing_df[missing_df.missing_count > 0].sort_values('missing_pct', ascending=False))

print("\n=== SAMPLE ROWS ===")
df.head(3)

## Section 3: GPS Coverage and Data Summary

In [ ]:
print("=== GPS COVERAGE ANALYSIS ===")

total = len(df)

# Rows with no latitude at all
no_lat = df['latitude'].isnull().sum()

# Rows where lat/lon exist but are (0, 0) — a common placeholder for "unknown"
zero_coords = ((df['latitude'] == 0) | (df['longitude'] == 0)).sum()

# Rows that are actually usable for spatial analysis
usable = df['latitude'].notnull() & (df['latitude'] != 0) & (df['longitude'] != 0)
usable_count = usable.sum()

print(f"Total rows:              {total:,}")
print(f"Missing latitude:        {no_lat:,}  ({no_lat/total*100:.1f}%)")
print(f"Zero coordinates:        {zero_coords:,}  ({zero_coords/total*100:.1f}%)")
print(f"Usable for spatial join: {usable_count:,}  ({usable_count/total*100:.1f}%)")

print("\n=== BOROUGH DISTRIBUTION ===")
print(df['borough'].value_counts(dropna=False))

print("\n=== DATE RANGE ===")
df['crash_date'] = pd.to_datetime(df['crash_date'])
print(f"Earliest: {df['crash_date'].min().date()}")
print(f"Latest:   {df['crash_date'].max().date()}")
print(f"Span:     {(df['crash_date'].max() - df['crash_date'].min()).days} days")

print("\n=== INJURY/FATALITY SUMMARY ===")
print(df[['number_of_persons_injured', 'number_of_persons_killed']].describe())

# Data Collection

This notebook handles data acquisition and initial loading.

In [ ]:
import pandas as pd
import numpy as np

print("Data collection notebook initialized")

## Load Raw Data

Load the raw crash data from the data/raw directory

In [ ]:
# Load data from CSV file
# df = pd.read_csv('../data/raw/nyc_crashes_raw.csv')
# print(f"Data shape: {df.shape}")
# df.head()